# Big Data con PySpark
## Análisis del Dataset de Accidentes en EE.UU. (US Accidents)
---
### Integrantes y Objetivos:
- **ETAPA 1:** Sesión Spark y lectura (nivel básico)
- **ETAPA 2:** Consultas y limpieza básica

## ETAPA 1: Sesión Spark y Lectura (Nivel Básico)
1. Instalación y verificación de PySpark.
2. Creación de la `SparkSession` en modo local.
3. Lectura del dataset con `header=True` e `inferSchema=True`.
4. Mostrar el esquema (`printSchema()`).
5. Conteo de filas del dataset.
6. Mostrar las primeras 5 filas (`show(5)`).
7. Limpieza y estandarización de nombres de columnas (eliminación de espacios y caracteres especiales).

In [1]:
# 1. Importación de módulos y creación de la SparkSession en modo local
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, desc, asc, isnan

spark = SparkSession.builder \
    .appName("Taller_BigData_US_Accidents_Etapas_1_2") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Configurar nivel de logs para una salida limpia
spark.sparkContext.setLogLevel("ERROR")
print(f"SparkSession creada exitosamente. Versión: {spark.version}")

c:\Users\JGuerra\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


SparkSession creada exitosamente. Versión: 4.2.0


In [2]:
# 2. Lectura del dataset con header=True e inferSchema=True
dataset_path = os.path.join("taller_bigData_PySpark", "US_Accidents_March23.csv")
if not os.path.exists(dataset_path):
    dataset_path = "US_Accidents_March23.csv"

print(f"Cargando dataset desde: {dataset_path}...")
df_raw = spark.read.csv(dataset_path, header=True, inferSchema=True)
print("Lectura completada.")

Cargando dataset desde: taller_bigData_PySpark\US_Accidents_March23.csv...
Lectura completada.


In [3]:
# 3. Mostrar el esquema del DataFrame
print("Esquema de datos:")
df_raw.printSchema()

Esquema de datos:
root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nulla

In [4]:
# 4. Conteo de filas y columnas originales
total_filas_original = df_raw.count()
total_columnas_original = len(df_raw.columns)
print(f"Total de filas en el dataset: {total_filas_original:,}")
print(f"Total de columnas: {total_columnas_original}")

Total de filas en el dataset: 7,728,394
Total de columnas: 46


In [5]:
# 5. Mostrar las primeras 5 filas
df_raw.show(5, truncate=True)

+---+-------+--------+-------------------+-------------------+-----------------+------------------+-------+-------+------------+--------------------+--------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
| ID| Source|Severity|         Start_Time|           End_Time|        Start_Lat|         Start_Lng|End_Lat|End_Lng|Distance(mi)|         Description|              Street|        City|    County|State|   Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition|Ameni

In [6]:
# 6. Limpieza y normalización de nombres de columnas (espacios y caracteres especiales)
df = df_raw
columnas_modificadas = {}

for columna in df.columns:
    nombre_limpio = columna.strip().replace(" ", "_")
    nombre_limpio = (
        nombre_limpio
        .replace("(", "_")
        .replace(")", "")
        .replace("%", "pct")
        .replace("-", "_")
    )
    if nombre_limpio != columna:
        columnas_modificadas[columna] = nombre_limpio
        df = df.withColumnRenamed(columna, nombre_limpio)

print(f"Columnas renombradas ({len(columnas_modificadas)}):", columnas_modificadas)
print("\nLista final de columnas:", df.columns)

Columnas renombradas (8): {'Distance(mi)': 'Distance_mi', 'Temperature(F)': 'Temperature_F', 'Wind_Chill(F)': 'Wind_Chill_F', 'Humidity(%)': 'Humidity_pct', 'Pressure(in)': 'Pressure_in', 'Visibility(mi)': 'Visibility_mi', 'Wind_Speed(mph)': 'Wind_Speed_mph', 'Precipitation(in)': 'Precipitation_in'}

Lista final de columnas: ['ID', 'Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance_mi', 'Description', 'Street', 'City', 'County', 'State', 'Zipcode', 'Country', 'Timezone', 'Airport_Code', 'Weather_Timestamp', 'Temperature_F', 'Wind_Chill_F', 'Humidity_pct', 'Pressure_in', 'Visibility_mi', 'Wind_Direction', 'Wind_Speed_mph', 'Precipitation_in', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']


## ETAPA 2: Consultas y Limpieza Básica
1. **Consultas con `select` y `filter`:**
   - Consulta por igualdad (Severidad 4 en California).
   - Consulta por condiciones numéricas (temperatura de congelación y visibilidad reducida).
   - Consultas con `orderBy` + `limit` (Top 10 ciudades con más accidentes y Top 5 accidentes con mayor congestión).
2. **Revisión de Nulos (`isNull`):** Conteo de nulos por columnas clave y visualización de registros nulos.
3. **Datos Válidos:** Filtrado lógico + `na.drop()`.
4. **Comparación:** Filas originales vs filas limpias.

In [7]:
# 1.1 Consulta por Igualdad: Accidentes graves (Severity == 4) en California (State == 'CA')
df_igualdad = df.filter((col("Severity") == 4) & (col("State") == "CA")) \
                .select("ID", "Severity", "State", "City", "Start_Time", "Description")

print(f"Total accidentes Severity 4 en California: {df_igualdad.count():,}")
df_igualdad.show(5, truncate=False)

Total accidentes Severity 4 en California: 13,502
+------+--------+-----+-----------+-------------------+-----------------------------------------------------------------------------------------------------------------+
|ID    |Severity|State|City       |Start_Time         |Description                                                                                                      |
+------+--------+-----+-----------+-------------------+-----------------------------------------------------------------------------------------------------------------+
|A-1198|4       |CA   |San Jose   |2016-06-24 22:28:49|Accident on I-280 Southbound at Exits 3A 3B Bird Ave.                                                            |
|A-1902|4       |CA   |San Bruno  |2016-07-01 14:09:13|Accident on I-280 Southbound before Exits 43A 43B I-380. On the median.                                          |
|A-4144|4       |CA   |San Jose   |2016-07-25 14:23:33|Right lane blocked and queueing traffic due t

In [8]:
# 1.2 Consulta Numérica: Clima gélido (Temperature_F <= 32) y baja visibilidad (Visibility_mi <= 1.0)
df_clima_adverso = df.filter(
    (col("Temperature_F") <= 32.0) & 
    (col("Visibility_mi") <= 1.0)
).select("ID", "City", "State", "Temperature_F", "Visibility_mi", "Weather_Condition", "Severity")

print(f"Total accidentes bajo clima adverso: {df_clima_adverso.count():,}")
df_clima_adverso.show(5, truncate=True)

Total accidentes bajo clima adverso: 80,600
+----+--------+-----+-------------+-------------+-----------------+--------+
|  ID|    City|State|Temperature_F|Visibility_mi|Weather_Condition|Severity|
+----+--------+-----+-------------+-------------+-----------------+--------+
|A-41| Lebanon|   OH|         21.0|          0.8|       Light Snow|       2|
|A-44|Columbus|   OH|         25.0|          1.0|       Light Snow|       3|
|A-46|Columbus|   OH|         25.0|          1.0|       Light Snow|       3|
|A-48|Columbus|   OH|         24.8|          1.0|       Light Snow|       3|
|A-65| Carroll|   OH|         21.9|          0.8|       Light Snow|       2|
+----+--------+-----+-------------+-------------+-----------------+--------+
only showing top 5 rows


In [9]:
# 1.3A Top 10 Ciudades con más accidentes (orderBy + limit)
top_ciudades = df.filter(col("City").isNotNull()) \
                 .groupBy("City", "State") \
                 .count() \
                 .orderBy(desc("count")) \
                 .limit(10)

top_ciudades.show(truncate=False)

+-----------+-----+------+
|City       |State|count |
+-----------+-----+------+
|Miami      |FL   |186768|
|Houston    |TX   |169428|
|Los Angeles|CA   |156491|
|Charlotte  |NC   |138345|
|Dallas     |TX   |130303|
|Orlando    |FL   |109690|
|Austin     |TX   |96604 |
|Raleigh    |NC   |86066 |
|Nashville  |TN   |72670 |
|Baton Rouge|LA   |71588 |
+-----------+-----+------+



In [10]:
# 1.3B Top 5 Accidentes con mayor afectación en distancia vial en millas (orderBy + limit)
top_distancias = df.select("ID", "City", "State", "Distance_mi", "Severity", "Start_Time", "Description") \
                   .orderBy(desc("Distance_mi")) \
                   .limit(5)

top_distancias.show(5, truncate=False)

+---------+------------+-----+------------------+--------+-------------------+----------------------------------------------------------------------------------------------------------+
|ID       |City        |State|Distance_mi       |Severity|Start_Time         |Description                                                                                               |
+---------+------------+-----+------------------+--------+-------------------+----------------------------------------------------------------------------------------------------------+
|A-1275668|Wells       |NV   |441.75            |3       |2020-11-09 07:48:27|Lane blocked due to accident on US-93 both ways from US-50 Great Basin Blvd to I-80 Exits 352 352A.       |
|A-575357 |Lake Charles|LA   |336.57000732421875|3       |2022-07-05 09:26:35|Right lane closed due to crash on I-10 Westbound from Exits 239A 239B Louisa St to Exit 29 Lake Shore Dr. |
|A-2640497|Walterboro  |SC   |333.6300048828125 |3       |2018-08-11 1

In [11]:
# 2. Revisión de Nulos (isNull)
columnas_analisis_nulos = [
    "City", "State", "Zipcode", "Temperature_F", 
    "Visibility_mi", "Wind_Speed_mph", "Precipitation_in", "Weather_Condition"
]

print("Conteo de valores nulos por columna clave:")
df_nulos = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in columnas_analisis_nulos
])
df_nulos.show()

print("Ejemplo de filas con nulos en City o Temperature_F:")
df.filter(col("City").isNull() | col("Temperature_F").isNull()) \
  .select("ID", "Street", "City", "State", "Temperature_F", "Weather_Condition") \
  .show(5, truncate=False)

Conteo de valores nulos por columna clave:
+----+-----+-------+-------------+-------------+--------------+----------------+-----------------+
|City|State|Zipcode|Temperature_F|Visibility_mi|Wind_Speed_mph|Precipitation_in|Weather_Condition|
+----+-----+-------+-------------+-------------+--------------+----------------+-----------------+
| 253|    0|   1915|       163853|       177098|        571233|         2203586|           173459|
+----+-----+-------+-------------+-------------+--------------+----------------+-----------------+

Ejemplo de filas con nulos en City o Temperature_F:
+------+----------------+-----------+-----+-------------+-----------------+
|ID    |Street          |City       |State|Temperature_F|Weather_Condition|
+------+----------------+-----------+-----+-------------+-----------------+
|A-602 |US Highway 250 N|Milan      |OH   |NULL         |NULL             |
|A-1735|Colony Rd       |Wilton     |CA   |NULL         |Clear            |
|A-1958|Tiburon Blvd    |Mill

In [12]:
# 3. Filtrado y Limpieza de Datos Válidos (filtro + na.drop)
# Columnas esenciales que no deben ser nulas
columnas_esenciales = ["City", "State", "Start_Time", "Severity", "Temperature_F", "Visibility_mi"]
df_sin_nulos = df.na.drop(subset=columnas_esenciales)

# Filtro lógico de negocio:
df_limpio = df_sin_nulos.filter(
    (col("Distance_mi") >= 0) &
    (col("Severity").between(1, 4)) &
    (col("Temperature_F").between(-60.0, 135.0)) &
    (col("Visibility_mi") >= 0.0)
)

In [13]:
# 4. Comparación de Filas: Originales vs Limpias
total_filas_limpias = df_limpio.count()
filas_eliminadas = total_filas_original - total_filas_limpias
porcentaje_retenido = (total_filas_limpias / total_filas_original) * 100
porcentaje_eliminado = (filas_eliminadas / total_filas_original) * 100

print("=" * 60)
print(f"Filas originales:           {total_filas_original:,}")
print(f"Filas limpias y válidas:    {total_filas_limpias:,}")
print(f"Filas eliminadas/filtradas: {filas_eliminadas:,} ({porcentaje_eliminado:.2f}%)")
print(f"Porcentaje retenido:        {porcentaje_retenido:.2f}%")
print("=" * 60)

df_limpio.select("ID", "Severity", "City", "State", "Temperature_F", "Visibility_mi", "Weather_Condition").show(5)

Filas originales:           7,728,394
Filas limpias y válidas:    7,524,687
Filas eliminadas/filtradas: 203,707 (2.64%)
Porcentaje retenido:        97.36%
+---+--------+------------+-----+-------------+-------------+-----------------+
| ID|Severity|        City|State|Temperature_F|Visibility_mi|Weather_Condition|
+---+--------+------------+-----+-------------+-------------+-----------------+
|A-1|       3|      Dayton|   OH|         36.9|         10.0|       Light Rain|
|A-2|       2|Reynoldsburg|   OH|         37.9|         10.0|       Light Rain|
|A-3|       2|Williamsburg|   OH|         36.0|         10.0|         Overcast|
|A-4|       3|      Dayton|   OH|         35.1|          9.0|    Mostly Cloudy|
|A-5|       2|      Dayton|   OH|         36.0|          6.0|    Mostly Cloudy|
+---+--------+------------+-----+-------------+-------------+-----------------+
only showing top 5 rows


In [14]:
# Cerrar la sesión de Spark al terminar el taller
spark.stop()
print("Sesión de Spark finalizada exitosamente.")

Sesión de Spark finalizada exitosamente.
